# Spark Assignment – Retail Transactions
Business Intelligence and Data Engineering, MSc in Business Analytics

In this notebook I analyse about 1 million online retail transactions using PySpark.
Part 1 answers the Spark SQL questions and Part 2 clusters the customers with KMeans.

## Setup
First I create the Spark session. It runs locally on my laptop using all the CPU cores.
(Spark needs Java 17 or 21 installed.)

In [ ]:
# install pyspark and pandas only if they are missing
# (pandas is only used to show the results as nice tables; pandas 3 doesn't work with pyspark 4.0 yet, so <3)
import importlib.util, subprocess, sys
for module, package in [("pyspark", "pyspark"), ("pandas", "pandas<3")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, LongType, IntegerType,
                               DoubleType, StringType, BooleanType, TimestampType)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("retail-assignment")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")  # less noise in the output
pd.set_option("display.max_columns", None)


def to_table(df, n=20):
    """Shows the first n rows of a Spark dataframe as a pandas table.
    Dates are turned into text first, because toPandas() can crash on them (timezone problem)."""
    for name, dtype in df.dtypes:
        if dtype == "timestamp":
            df = df.withColumn(name, F.date_format(name, "yyyy-MM-dd HH:mm:ss"))
    return df.limit(n).toPandas()


spark.version

## Loading the data
I give the schema myself instead of using `inferSchema`, so Spark doesn't need an extra pass over the file
and the `Date` column is read as a timestamp. Change `DATA_PATH` if the csv is somewhere else.

In [2]:
import os, zipfile

DATA_PATH = "retail-dataset.csv"   # put the csv in the same folder as the notebook

# the dataset was given as a zip, so if only the zip is there I extract the csv from it
if not os.path.exists(DATA_PATH) and os.path.exists(DATA_PATH + ".zip"):
    with zipfile.ZipFile(DATA_PATH + ".zip") as z:
        csv_name = [n for n in z.namelist() if n.endswith(".csv") and "__MACOSX" not in n][0]
        DATA_PATH = z.extract(csv_name, ".")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Put retail-dataset.csv (or the .zip) next to this notebook or change DATA_PATH")

schema = StructType([
    StructField("Transaction_ID", LongType()),
    StructField("Date", TimestampType()),
    StructField("Customer_Name", StringType()),
    StructField("Product", StringType()), # it's a list but stored as text
    StructField("Total_Items", IntegerType()),
    StructField("Total_Cost", DoubleType()),
    StructField("Payment_Method", StringType()),
    StructField("City", StringType()),
    StructField("Store_Type", StringType()),
    StructField("Discount_Applied", BooleanType()),
    StructField("Customer_Category", StringType()),
    StructField("Season", StringType()),
    StructField("Promotion", StringType()),
])

retail = spark.read.csv(DATA_PATH, header=True, schema=schema)
retail.printSchema()

root
 |-- Transaction_ID: long (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Total_Items: integer (nullable = true)
 |-- Total_Cost: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Discount_Applied: boolean (nullable = true)
 |-- Customer_Category: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Promotion: string (nullable = true)



I cache the dataframe because I will query it many times below.

In [ ]:
retail.cache()
print("rows:", retail.count())
to_table(retail, 5)

## Part 1 – Spark SQL

### Q1. Descriptive statistics for each column
`describe()` gives count, mean, stddev, min and max for every column. It only works with numbers and text,
so Spark skips `Date` and `Discount_Applied` here and I look at those two separately below.

In [ ]:
to_table(retail.describe())

For the two numeric columns I also use `summary()`, which adds the quartiles (25%, 50%, 75%):

In [ ]:
to_table(retail.select("Total_Items", "Total_Cost").summary())

The date range and the true/false split of `Discount_Applied`:

In [ ]:
display(to_table(retail.select(F.min("Date").alias("first_date"), F.max("Date").alias("last_date"))))
display(to_table(retail.groupBy("Discount_Applied").count()))

Finally, how many distinct values and how many missing (null) values each column has:

In [ ]:
# distinct values per column
display(to_table(retail.select([F.countDistinct(c).alias(c) for c in retail.columns])))

# missing values per column
display(to_table(retail.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in retail.columns])))

**Comments.** There are no missing values in any column. The data covers 1,000,000 transactions from January 2020
to May 2024, in 10 cities, 6 store types, 8 customer categories and 4 payment methods. Each purchase has 1–10 items
and costs between 5 and 100 (about 52 on average). The mean and stddev of `Transaction_ID` don't mean anything since
it is just an id. Also, in `Promotion` "no promotion" is written as the text `None` and not as a real null,
which I need to remember for Q5.